#Boca Raton Inlet


In [1]:
# Install required packages
!pip install ultralytics sahi pycocotools -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.3 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
from collections import defaultdict

from ultralytics import YOLO
from sahi.predict import get_sliced_prediction
from sahi import AutoDetectionModel
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [22]:
#File paths
MODEL_PATH = "yolo11x.pt"
IMAGE_FOLDER = "/content/Boca_Raton"
GT_JSON = "/content/_annotations.coco.json"
OUTPUT_FOLDER = "/content/Boca_Raton/output_improved"
DEVICE = "cuda:0"  # Use GPU

#Setting the parameters
SAHI_SLICE_SIZE = 64    #~9 slices per image
SAHI_OVERLAP = 0.3
CONF_THRESHOLD = 0.15

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print("Configuration loaded!")
print(f"  Slice size: {SAHI_SLICE_SIZE}×{SAHI_SLICE_SIZE}")
print(f"  Overlap: {SAHI_OVERLAP}")
print(f"  Confidence threshold: {CONF_THRESHOLD}")

Configuration loaded!
  Slice size: 64×64
  Overlap: 0.3
  Confidence threshold: 0.15


## Load and Analyze Ground Truth

In [23]:
# Load ground truth
with open(GT_JSON) as f:
    gt = json.load(f)

print(f"Total images: {len(gt['images'])}")
print(f"Total annotations: {len(gt['annotations'])}")

# Analyze boat sizes - this shows WHY small object detection matters
boat_annotations = [ann for ann in gt['annotations'] if ann['category_id'] == 1]
boat_areas = [ann['bbox'][2] * ann['bbox'][3] for ann in boat_annotations]
boat_widths = [ann['bbox'][2] for ann in boat_annotations]
boat_heights = [ann['bbox'][3] for ann in boat_annotations]

print(f"\nBoca Raton Boats:")
print(f"  Total boats: {len(boat_areas)}")
print(f"  \n  Bounding box areas (pixels²):")
print(f"    Min: {min(boat_areas):.0f}, Max: {max(boat_areas):.0f}, Mean: {np.mean(boat_areas):.0f}")
print(f"  \n  Widths (pixels):")
print(f"    Min: {min(boat_widths):.0f}, Max: {max(boat_widths):.0f}, Mean: {np.mean(boat_widths):.0f}")

# COCO size categories
small = len([a for a in boat_areas if a < 32*32])    # < 1024 px²
medium = len([a for a in boat_areas if 32*32 <= a < 96*96])  # 1024-9216 px²
large = len([a for a in boat_areas if a >= 96*96])   # >= 9216 px²

print(f"\n  COCO Size Categories:")
print(f"    Small (<32×32):  {small} boats ({small/len(boat_areas)*100:.1f}%)")
print(f"    Medium (32-96):  {medium} boats ({medium/len(boat_areas)*100:.1f}%)")
print(f"    Large (>96×96):  {large} boats ({large/len(boat_areas)*100:.1f}%)")


Total images: 46
Total annotations: 197

Boca Raton Boats:
  Total boats: 169
  
  Bounding box areas (pixels²):
    Min: 1, Max: 12710, Mean: 299
  
  Widths (pixels):
    Min: 1, Max: 114, Mean: 9

  COCO Size Categories:
    Small (<32×32):  161 boats (95.3%)
    Medium (32-96):  7 boats (4.1%)
    Large (>96×96):  1 boats (0.6%)


In [24]:
from PIL import Image
img = Image.open("/content/Boca_Raton/image-20230703131727_jpg.rf.485cdc4508451c77b4ce2044b9f50f36.jpg")
img_width, img_height = img.size
img.size


import json, numpy as np
from PIL import Image
import os

def get_optimal_sahi_params(image_folder, coco_json, category_id=1):
    # Get image size
    imgs = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]
    w, h = Image.open(os.path.join(image_folder, imgs[0])).size

    # Get object sizes
    with open(coco_json) as f:
        coco = json.load(f)
    anns = [a for a in coco['annotations'] if a['category_id'] == category_id]
    diagonals = [np.sqrt(a['bbox'][2]**2 + a['bbox'][3]**2) for a in anns]
    median_diag = np.median(diagonals)

    # Calculate slice size (objects should be ~30% of slice)
    slice_size = int((median_diag / 0.30) // 32) * 32
    slice_size = max(128, min(slice_size, min(w, h)))

    print(f"Image: {w}x{h}, Median object diagonal: {median_diag:.0f}px")
    print(f"→ Recommended slice_size: {slice_size}")
    return slice_size

#Now applying the function
slice_size = get_optimal_sahi_params("/content/Boca_Raton",
                                      "/content/_annotations.coco.json")

Image: 640x640, Median object diagonal: 5px
→ Recommended slice_size: 128


## Method 1: Standard YOLO

In [5]:
print("="*60)
print("METHOD 1: Standard YOLO (Baseline)")
print("="*60)

model = YOLO(MODEL_PATH)

predictions_baseline = []

for img_data in tqdm(gt['images'], desc="Standard YOLO"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue

    results = model(img_path, conf=0.3, device=DEVICE, verbose=False)  # Original conf=0.3

    for result in results:
        boxes = result.boxes
        for i in range(len(boxes)):
            class_id = int(boxes.cls[i])
            if class_id == 8:  # YOLO boat class
                xyxy = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i])
                predictions_baseline.append({
                    "image_id": img_data['id'],
                    "category_id": 1,
                    "bbox": [float(xyxy[0]), float(xyxy[1]),
                            float(xyxy[2] - xyxy[0]), float(xyxy[3] - xyxy[1])],
                    "score": conf
                })

print(f"\nBaseline predictions: {len(predictions_baseline)}")

# Evaluate
coco_gt = COCO(GT_JSON)
if predictions_baseline:
    coco_dt = coco_gt.loadRes(predictions_baseline)
    coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
    coco_eval.params.catIds = [1]
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

METHOD 1: Standard YOLO (Baseline)


Standard YOLO: 100%|██████████| 46/46 [00:04<00:00,  9.27it/s]



Baseline predictions: 35
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.06s).
Accumulating evaluation results...
DONE (t=0.04s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.065
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.102
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.067
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.048
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.481
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.057
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.079
 Average Recall     (AR) @[ IoU=0.50:0.

## Method 2: SAHI

In [ ]:
print("="*60)
print("METHOD 2:SAHI Detection")
print("="*60)
print(f"Slice size: {SAHI_SLICE_SIZE}×{SAHI_SLICE_SIZE}")
print(f"Overlap: {SAHI_OVERLAP}")
print(f"Confidence: {CONF_THRESHOLD}")

# Calculate expected slices for a 640x640 image
stride = SAHI_SLICE_SIZE * (1 - SAHI_OVERLAP)
expected_slices = ((640 - SAHI_SLICE_SIZE) / stride + 1) ** 2
print(f"\nExpected slices per image: ~{expected_slices:.0f}")

detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=MODEL_PATH,
    confidence_threshold=CONF_THRESHOLD,
    device=DEVICE
)

predictions_sahi = []

for img_data in tqdm(gt['images'], desc="SAHI Detection"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue

    result = get_sliced_prediction(
        img_path,
        detection_model,
        slice_height=SAHI_SLICE_SIZE,
        slice_width=SAHI_SLICE_SIZE,
        overlap_height_ratio=SAHI_OVERLAP,
        overlap_width_ratio=SAHI_OVERLAP,
        postprocess_type="NMS",
        postprocess_match_metric="IOS",
        postprocess_match_threshold=0.5,
    )

    for obj in result.object_prediction_list:
        if obj.category.id == 8:  # YOLO boat class
            bbox = obj.bbox
            predictions_sahi.append({
                "image_id": img_data['id'],
                "category_id": 1,
                "bbox": [bbox.minx, bbox.miny, bbox.maxx - bbox.minx, bbox.maxy - bbox.miny],
                "score": obj.score.value
            })

print(f"\nSAHI predictions: {len(predictions_sahi)}")
print(f"Improvement: {len(predictions_sahi) - len(predictions_baseline):+d} detections")

# Evaluate
if predictions_sahi:
    coco_dt_sahi = coco_gt.loadRes(predictions_sahi)
    coco_eval_sahi = COCOeval(coco_gt, coco_dt_sahi, 'bbox')
    coco_eval_sahi.params.catIds = [1]
    coco_eval_sahi.evaluate()
    coco_eval_sahi.accumulate()
    coco_eval_sahi.summarize()

METHOD 2:SAHI Detection
Slice size: 64×64
Overlap: 0.3
Confidence: 0.15

Expected slices per image: ~192


SAHI Detection:   0%|          | 0/46 [00:00<?, ?it/s]

Performing prediction on 196 slices.


SAHI Detection:   2%|▏         | 1/46 [00:12<09:18, 12.41s/it]

Performing prediction on 196 slices.


...

Performing prediction on 196 slices.


SAHI Detection:  11%|█         | 5/46 [01:02<08:29, 12.43s/it]

Performing prediction on 196 slices.


SAHI Detection:  13%|█▎        | 6/46 [01:14<08:18, 12.47s/it]

Performing prediction on 196 slices.


## Method 3: Multi-Scale Detection


In [14]:
print("="*60)
print("METHOD 3: Multi-Scale Detection")
print("="*60)

def detect_multiscale(model, image_path, scales=[1.0, 1.5, 2.0], conf=0.2):
    """Detect at multiple scales and merge results."""
    img = cv2.imread(image_path)
    orig_h, orig_w = img.shape[:2]

    all_boxes = []
    all_scores = []

    for scale in scales:
        if scale == 1.0:
            results = model(image_path, conf=conf, verbose=False)
        else:
            # Upscale image
            scaled = cv2.resize(img, None, fx=scale, fy=scale,
                              interpolation=cv2.INTER_LANCZOS4)
            temp_path = "/tmp/scaled_temp.jpg"
            cv2.imwrite(temp_path, scaled)
            results = model(temp_path, conf=conf, verbose=False)

        for result in results:
            boxes = result.boxes
            for i in range(len(boxes)):
                if int(boxes.cls[i]) == 8:  # boat
                    xyxy = boxes.xyxy[i].cpu().numpy()
                    if scale != 1.0:
                        xyxy = xyxy / scale  # Scale back
                    all_boxes.append(xyxy)
                    all_scores.append(float(boxes.conf[i]))

    # Apply NMS
    if not all_boxes:
        return []

    boxes_arr = np.array(all_boxes)
    scores_arr = np.array(all_scores)

    # Simple NMS
    keep = []
    order = scores_arr.argsort()[::-1]

    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break

        xx1 = np.maximum(boxes_arr[i, 0], boxes_arr[order[1:], 0])
        yy1 = np.maximum(boxes_arr[i, 1], boxes_arr[order[1:], 1])
        xx2 = np.minimum(boxes_arr[i, 2], boxes_arr[order[1:], 2])
        yy2 = np.minimum(boxes_arr[i, 3], boxes_arr[order[1:], 3])

        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        inter = w * h

        areas = (boxes_arr[order[1:], 2] - boxes_arr[order[1:], 0]) * \
                (boxes_arr[order[1:], 3] - boxes_arr[order[1:], 1])
        area_i = (boxes_arr[i, 2] - boxes_arr[i, 0]) * (boxes_arr[i, 3] - boxes_arr[i, 1])

        iou = inter / (area_i + areas - inter + 1e-6)
        inds = np.where(iou <= 0.5)[0]
        order = order[inds + 1]

    results = []
    for idx in keep:
        xyxy = boxes_arr[idx]
        results.append({
            'bbox': [float(xyxy[0]), float(xyxy[1]),
                    float(xyxy[2]-xyxy[0]), float(xyxy[3]-xyxy[1])],
            'score': float(scores_arr[idx])
        })
    return results

predictions_multiscale = []

for img_data in tqdm(gt['images'], desc="Multi-scale Detection"):
    img_path = os.path.join(IMAGE_FOLDER, img_data['file_name'])
    if not os.path.exists(img_path):
        continue

    detections = detect_multiscale(model, img_path)
    for det in detections:
        predictions_multiscale.append({
            "image_id": img_data['id'],
            "category_id": 1,
            "bbox": det['bbox'],
            "score": det['score']
        })

print(f"\nMulti-scale predictions: {len(predictions_multiscale)}")

if predictions_multiscale:
    coco_dt_ms = coco_gt.loadRes(predictions_multiscale)
    coco_eval_ms = COCOeval(coco_gt, coco_dt_ms, 'bbox')
    coco_eval_ms.params.catIds = [1]
    coco_eval_ms.evaluate()
    coco_eval_ms.accumulate()
    coco_eval_ms.summarize()

METHOD 3: Multi-Scale Detection


Multi-scale Detection: 100%|██████████| 46/46 [00:12<00:00,  3.75it/s]


Multi-scale predictions: 42
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.070
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.108
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.077
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.053
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.477
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.700
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.062
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.087
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.087
 Average Recall     (AR) @[ IoU=0.50:0.

## Method 4: Ensemble (SAHI + Multi-scale)

In [15]:
print("="*60)
print("METHOD 4: Ensemble (SAHI + Multi-scale)")
print("="*60)

# Combine predictions
combined_by_image = defaultdict(list)

for pred in predictions_sahi:
    combined_by_image[pred['image_id']].append(pred)

for pred in predictions_multiscale:
    combined_by_image[pred['image_id']].append(pred)

# Apply NMS per image
predictions_ensemble = []

for img_id, preds in combined_by_image.items():
    if not preds:
        continue

    boxes = np.array([[p['bbox'][0], p['bbox'][1],
                      p['bbox'][0]+p['bbox'][2], p['bbox'][1]+p['bbox'][3]]
                     for p in preds])
    scores = np.array([p['score'] for p in preds])

    # NMS
    keep = []
    order = scores.argsort()[::-1]

    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break

        xx1 = np.maximum(boxes[i, 0], boxes[order[1:], 0])
        yy1 = np.maximum(boxes[i, 1], boxes[order[1:], 1])
        xx2 = np.minimum(boxes[i, 2], boxes[order[1:], 2])
        yy2 = np.minimum(boxes[i, 3], boxes[order[1:], 3])

        w = np.maximum(0, xx2 - xx1)
        h = np.maximum(0, yy2 - yy1)
        inter = w * h

        areas = (boxes[order[1:], 2] - boxes[order[1:], 0]) * \
                (boxes[order[1:], 3] - boxes[order[1:], 1])
        area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])

        iou = inter / (area_i + areas - inter + 1e-6)
        inds = np.where(iou <= 0.5)[0]
        order = order[inds + 1]

    for idx in keep:
        predictions_ensemble.append(preds[idx])

print(f"\nEnsemble predictions: {len(predictions_ensemble)}")

if predictions_ensemble:
    coco_dt_ens = coco_gt.loadRes(predictions_ensemble)
    coco_eval_ens = COCOeval(coco_gt, coco_dt_ens, 'bbox')
    coco_eval_ens.params.catIds = [1]
    coco_eval_ens.evaluate()
    coco_eval_ens.accumulate()
    coco_eval_ens.summarize()

METHOD 4: Ensemble (SAHI + Multi-scale)

Ensemble predictions: 63
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.02s).
Accumulating evaluation results...
DONE (t=0.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.067
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.122
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.072
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.064
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.297
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.200
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.069
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.100
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.100
 A

## Results Comparison

In [18]:
print("\n" + "="*70)
print("RESULTS COMPARISON")
print("="*70)
print(f"\n{'Method':<25} {'Predictions':>12} {'vs GT (169)':>15}")
print("-"*55)
print(f"{'Standard YOLO':<25} {len(predictions_baseline):>12} {len(predictions_baseline)/169*100:>14.1f}%")
print(f"{'SAHI':<25} {len(predictions_sahi):>12} {len(predictions_sahi)/169*100:>14.1f}%")
print(f"{'Multi-scale':<25} {len(predictions_multiscale):>12} {len(predictions_multiscale)/169*100:>14.1f}%")
print(f"{'Ensemble':<25} {len(predictions_ensemble):>12} {len(predictions_ensemble)/169*100:>14.1f}%")
print("-"*55)
print(f"\nGround Truth boats: 169")


RESULTS COMPARISON

Method                     Predictions     vs GT (169)
-------------------------------------------------------
Standard YOLO                       35           20.7%
SAHI                                63           37.3%
Multi-scale                         42           24.9%
Ensemble                            63           37.3%
-------------------------------------------------------

Ground Truth boats: 169


## Visualize Results

In [ ]:
# Create visualization folder
vis_folder = os.path.join(OUTPUT_FOLDER, "visualizations")
os.makedirs(vis_folder, exist_ok=True)

# Group predictions by image
pred_by_image = defaultdict(list)
for pred in predictions_ensemble:  # Use ensemble (best) results
    pred_by_image[pred['image_id']].append(pred)

# Group GT by image
gt_by_image = defaultdict(list)
for ann in gt['annotations']:
    if ann['category_id'] == 1:
        gt_by_image[ann['image_id']].append(ann)

id_to_filename = {img['id']: img['file_name'] for img in gt['images']}

# Visualize first 10 images with GT boats
for img_id in list(gt_by_image.keys())[:10]:
    img_path = os.path.join(IMAGE_FOLDER, id_to_filename[img_id])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Draw GT in GREEN
    for ann in gt_by_image[img_id]:
        x, y, w, h = [int(v) for v in ann['bbox']]
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img, 'GT', (x, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Draw predictions in RED
    for pred in pred_by_image[img_id]:
        x, y, w, h = [int(v) for v in pred['bbox']]
        cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img, f"{pred['score']:.2f}", (x, y-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.title(f"{id_to_filename[img_id]}\nGT: {len(gt_by_image[img_id])} (green), Pred: {len(pred_by_image[img_id])} (red)")
    plt.axis('off')
    plt.savefig(os.path.join(vis_folder, id_to_filename[img_id]))
    plt.show()

print(f"\nVisualizations saved to: {vis_folder}")

## Save Best Predictions

In [ ]:
# Save the best predictions (ensemble)
pred_json_path = os.path.join(OUTPUT_FOLDER, "predictions_improved.json")
with open(pred_json_path, 'w') as f:
    json.dump(predictions_ensemble, f, indent=2)

print(f"Predictions saved to: {pred_json_path}")
print(f"Total predictions: {len(predictions_ensemble)}")